# Applet interactiva (Adri)

Las celdas que empiezan por `# PRUEBA` **no** se cargan en la web; úsalas para probar.

In [ ]:
# PRUEBA
# Carga el algoritmo de Lola para poder probar la applet aquí
%run melkman.ipynb

In [1]:
def nombre(v, P):
    """
    Input: v (vértice), P (vértices del polígono)
    Output: nombre del vértice ('v0', 'v1', ...)
    """
    return "v%d" % P.index(v)

In [5]:
def estado_previo(pasos, k):
    """
    Input: pasos(lista de melkman), k (nº de paso, debe ser k>=1)
    Output: C (cierre antes del paso k)
            q (último vértice añadido al cierre antes del paso k)
    
    En el paso k se comprueba si el vértice p está en R_q, usando el cierre y el q que había justo antes.
    La variable 'pasos', guarda el cierre DESPUÉS de cada paso, así que hay que mirar el paso anterior
    """
    C = pasos[k-1]['C']
    j = k-1
    while pasos[j]['caso'] == 1: #los descartados no cambian q
        j -= 1

    return C, pasos[j]['p']

In [4]:
def region_Rq(q, C, O, L):
    """
    Input: q (vértice del cierre), C (cierre), O (orientación), L (longitud de los lados)
    Output: polígono grande que representa la región R_q
    
    R_q es el ángulo con vértice q, formado por las rectas que pasan por q y sus dos vecinos en C (ANTERIOR y POSTERIOR).
    Si un vértice cae ahí, no puede estar en el cierre (es el mismo caso que en_Rq)
    Si q y sus vecinos etán alineados, entonces R_q será un semiplano
    """

    #definimos a y b, vecinos de q en C
    a, b = ANTERIOR(q, C), POSTERIOR(q, C)
    ra = (q[0]+ L*(a[0]-q[0]), q[1] + L*(a[1]-q[1]))
    rb = (q[0]+ L*(b[0]-q[0]), q[1] + L*(b[1]-q[1]))
    if signo(a, q, b) != 0:
        return [q, ra, rb]

    #Caso alineado: rectángulo hacia el lado de R_q
    n = (-(b[1] - q[1]), b[0]-q[0]) #perpendicular a q: b
    if O*signo(q, b, (q[0]+n[0], q[1]+n[1])) <0:
        n = (-n[0], -n[1])
    return [ra, rb, (rb[0] + L*n[0], rb[1]+ L*n[1]), (ra[0]+L*n[0], ra[1]+ L*n[1])]


In [7]:
def describir_paso(P, pasos, k):
    """
    Input: P(vértices del polígono), pasos (lista de melkman), k (nº de paso)
    Output: texto con el estado del algoritmo en el paso k (abordamos distintos casos)
    """

    N = len(pasos)
    paso = pasos[k]
    p = nombre(paso['p'], P)
    deque = lambda C: " ".join(nombre(v, P) for v in C)

    lineas = ["Paso %d de %d" % (k, N-1)]
    if paso['caso'] == 'inicio': #caso inicial
        lineas.append("Triángulo inicial con v0, v1 y v2")
    else:
        C_antes, q = estado_previo(pasos, k)
        lineas.append("Deque C antes:  " + deque(C_antes))
        lineas.append("q (último añadido):  " + nombre(q, P))
        if paso['caso'] == 1:
            lineas.append(p + " ∈ R_q -> se descarta (no puede estar en el cierre)")
        else:
            lineas.append(p + " ∉ R_q -> entra en el cierre")
            if paso['borrados']:
                lineas.append("Se eliminan: " + ", ".join(nombre(b, P) for b in paso['borrados']))

    lineas.append("Deque C después: " + deque(paso['C']))
    return "\n".join(lineas)

In [ ]:
def dibujar_paso(P, pasos, k):
    """
    Input: P (vértices del polígono), pasos (lista de melkman), k (nº de paso)
    Output: gráfico de Sage con el estado del algoritmo en el paso k

    Morado: región R_q con la que se comprueba el vértice del paso
    Gris claro: parte del polígono que aún no se ha leído
    Negro: parte ya recorrida
    Verde: cierre convexo después del paso
    Rojo/azul: vértice del paso (descartado/añadido)
    Naranja: vértices eliminados del cierre en este paso
    """
    paso = pasos[k]
    p, caso, C, borrados = paso['p'], paso['caso'], paso['C'], paso['borrados']
    i = P.index(p) #se han leído los vértices v0,...vi

    #Marco del dibujo y separación de las etiqeutas 
    xs = [v[0] for v in P]
    ys = [v[1] for v in P]
    tam = max(max(xs) - min(xs), max(ys)-min(ys))
    m, d = 0.1*tam, 0.03*tam 
    marco = dict(xmin=min(xs)-m, xmax=max(xs)+m, ymin=min(ys)-m, ymax=max(ys)+m)

    G = Graphics()
    if caso != 'inicio':
        C_antes, q = estado_previo(pasos, k)
        O = signo(P[0], P[1], P[2]) #orientación del triángulo inicial
        G += polygon(region_Rq(q, C_antes, O, 100), color='purple', alpha=0.15) #R_q
        G += point(q, color='purple', size=60)
    G += polygon(P, fill=False, color='lightgray', linestyle='--') #polígono completo
    G += line(P[:i+1], color='black', thickness=1.5) #parte recorrida
    G += polygon(C, color = 'green', alpha=0.25) #cierre actual
    G += line(C + [C[0]], color='green', thickness=2.5)
    G += points(P[i+1:], color='lightgray', size=25) #vértices sin leer
    G += points(P[:i+1], color='black', size=25) #vértices leídos

    if borrados:
        G+= points(borrados, color='orange', size=120, marker='x')
    color = {'inicio': 'purple', 1: 'red', 2: 'blue'}[caso]
    G += point(p, color=color, size=90)
    for j, v in enumerate(P):
        G += text("v%d" % j, (v[0] + d, v[1] + d), color='gray', fontsize=9)
    G.set_axes_range(**marco) #recorta R_q al marco del polígono 
    return G

In [ ]:
@interact
def applet(poligono=selector(list(ejemplos.keys()) + ['Personalizado'],
            label = 'Polígono', buttons=False),
            coords = input_box(default=[(0,0), (4,0), (4,3), (2,1), (0,3)],
                               label="Coordenadas (para Personalizado)", width=60),
            paso=slider(0, 60, 1, 0, label="Paso")):
    #1. Polígono elegido (en decimales para evitar cálculos simbólicos lentos)
    P = coords if poligono == "Personalizado" else ejemplos[poligono]
    P = [(float(x), float(y)) for (x,y) in P]

    #2. Comprobaciones
    if len(P) <3:
        print("El polígono necesita al menos 3 vértices")
        return 
    if len(set(P)) < len(P):
        print("Hay vértices repetidos: cada vértice debe aparecer una sola vez")
        return 

    if signo(P[0], P[1], P[2])==0:
        print("Los tres primeros vértices están alineados: empieza el polígono por otro vértice.")
        return

    #3. Algoritmo de melkman
    casco, pasos = melkman(P)
    k = min(paso, len(pasos)-1) #si el deslizador se pasa, se muestra el último paso

    #4. Panel de estado y dibujo
    print(describir_paso(P, pasos, k))
    if k == len(pasos)-1:
        print("\nFin: el cierre convexo tiene %d vértices (de %d del polígono)" % (len(casco), len(P)))
    show(dibujar_paso(P, pasos, k), aspect_ratio=1, axes=False, figsize=6)